In [1]:
import cmocean
import numpy as np 
import xarray as xr
import pandas as pd 
import seaborn as sns
import cartopy.crs as ccrs
import statsmodels.api as sm
import matplotlib.pyplot as plt
from IPython.display import HTML
from scipy.stats import linregress 
from nemo_cookbook import NEMODataTree 
from matplotlib.patches import Rectangle
from OceanDataStore import OceanDataCatalog 
from matplotlib.animation import FuncAnimation


In [2]:
# Open model and config files
catalog = OceanDataCatalog(catalog_name="noc-model-stac")
catalog.search(collection='noc-npd-era5', standard_name='sea_surface_temperature')
ds1 = catalog.open_dataset(id=catalog.Items[2].id,
                          start_datetime='1990-01',
                          end_datetime='2024-12')
catalog.search(collection='noc-npd-era5', item_name='domain_cfg')
config = catalog.open_dataset(id=catalog.Items[1].id)

# Merge into a data tree
datasets = {"parent": {"domain": config, "gridT": ds1}}
dt_global = NEMODataTree.from_datasets(datasets = datasets)

# Clip to North Atlantic 
bbox = (-85.0, 0.0, 0.0, 80.0)
dt_clipped = dt_global.clip_grid(grid="gridT", bbox=bbox)

# Add lat and lon as co-ordinates
dt = dt_clipped.add_geoindex(grid="gridT")

# Correct compatibility error 
dt['gridT']['tmaskutil'] = dt['gridT']['tmaskutil'].astype(bool)


# Compute areas
areas = dt.cell_area(grid="gridT", dim="k")

# Convert to dataset
ds = (dt['gridT']).dataset



            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1y
              Title: eORCA1 ERA5v1 NPD T1y Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics annual mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2025-07-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca1-era5v1/gn/T1m
              Title: eORCA1 ERA5v1 NPD T1m Icechunk repository
              Description: Icechunk repository containing eORCA1 ERA5v1 NPD global ocean physics monthly mean outputs defined at T-points.
              Platform: gn
              Start Date: 1976-01-01T00:00:00Z
              End Date: 2025-07-31T00:00:00Z
            

            * Item ID: noc-npd-era5/npd-eorca025-era5v1/gn/T1y_3d
              Title: eORCA025 ERA5v1 NPD T1y_3d Icechunk repository
              Description: Icechunk repository containing eORCA025 ERA5v1 

In [4]:
dt

<xarray.DataTree>
Group: /
│   Dimensions:               (time_counter: 35, axis_nbounds: 2)
│   Coordinates:
│     * time_counter          (time_counter) datetime64[ns] 280B 1990-07-02T12:00...
│       time_centered         (time_counter) datetime64[ns] 280B dask.array<chunksize=(1,), meta=np.ndarray>
│   Dimensions without coordinates: axis_nbounds
│   Data variables:
│       time_centered_bounds  (time_counter, axis_nbounds) datetime64[ns] 560B dask.array<chunksize=(1, 2), meta=np.ndarray>
│       time_counter_bounds   (time_counter, axis_nbounds) datetime64[ns] 560B dask.array<chunksize=(1, 2), meta=np.ndarray>
│   Attributes:
│       nftype:   None
│       iperio:   False
├── Group: /gridT
│       Dimensions:                (time_counter: 35, j: 482, i: 341, axis_nbounds: 2,
│                                   k: 75)
│       Coordinates:
│         * j                      (j) int64 4kB 685 686 687 688 ... 1163 1164 1165 1166
│         * i                      (i) int64 3kB 809 810 811 812 ... 1146 1147 1148 1149
│         * k                      (k) int64 600B 1 2 3 4 5 6 7 ... 69 70 71 72 73 74 75
│           time_centered          (time_counter) datetime64[ns] 280B dask.array<chunksize=(1,), meta=np.ndarray>
│         * gphit                  (j, i) float64 1MB 0.0 0.0 0.0 ... 79.41 79.3 79.2
│         * glamt                  (j, i) float64 1MB -85.0 -84.75 -84.5 ... 48.54 48.79
│       Dimensions without coordinates: axis_nbounds
│       Data variables: (12/57)
│           berg_latent_heat_flux  (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
│           empmr                  (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
│           evs                    (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
│           hfempds                (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
│           hfds                   (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
│           hfevapds               (time_counter, j, i) float32 23MB dask.array<chunksize=(1, 482, 341), meta=np.ndarray>
│           ...                     ...
│           e1t                    (j, i) float64 1MB dask.array<chunksize=(482, 341), meta=np.ndarray>
│           e2t                    (j, i) float64 1MB dask.array<chunksize=(482, 341), meta=np.ndarray>
│           top_level              (j, i) int32 657kB dask.array<chunksize=(482, 341), meta=np.ndarray>
│           bottom_level           (j, i) int32 657kB dask.array<chunksize=(482, 341), meta=np.ndarray>
│           tmask                  (k, j, i) float64 99MB nan 1.0 1.0 ... nan nan nan
│           tmaskutil              (j, i) bool 164kB True True True ... True True True
│       Indexes:
│         ┌ gphit    NDPointIndex (SklearnGeoBallTreeAdapter)
│         └ glamt
│       Attributes:
│           nftype:   None
│           iperio:   False
├── Group: /gridU
│       Dimensions:       (j: 1206, i: 1440, k: 75)
│       Coordinates:
│         * j             (j) int64 10kB 1 2 3 4 5 6 7 ... 1201 1202 1203 1204 1205 1206
│         * i             (i) float64 12kB 1.5 2.5 3.5 ... 1.438e+03 1.44e+03 1.44e+03
│         * k             (k) int64 600B 1 2 3 4 5 6 7 8 9 ... 68 69 70 71 72 73 74 75
│           gphiu         (j, i) float64 14MB dask.array<chunksize=(603, 720), meta=np.ndarray>
│           glamu         (j, i) float64 14MB dask.array<chunksize=(603, 720), meta=np.ndarray>
│       Data variables:
│           e1u           (j, i) float64 14MB dask.array<chunksize=(603, 720), meta=np.ndarray>
│           e2u           (j, i) float64 14MB dask.array<chunksize=(603, 720), meta=np.ndarray>
│           umask         (k, j, i) float64 1GB 0.0 0.0 0.0 0.0 0.0 ... nan nan nan nan
│           umaskutil     (j, i) float64 14MB 0.0 0.0 0.0 0.0 0.0 ... nan nan nan nan
│       Attributes:
│           nft

In [3]:
# Compute barotropic stream function:
bsf = dt.integral(grid="gridV", var="vo", dims=["k", "i"], cum_dims=["i"], dir="+1")
bsf_final = (1E-6 * bsf_atl).compute()


KeyError: "variable 'vo' not found in grid 'gridV'."